In [ ]:
import random
import gym
from gymnasium import spaces
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

Initializing parameters

In [ ]:
# Option parameters
Strike = 100
Maturity = 21*3/250

# Asset parameters
SpotPrice = 100
ExpVol = 0.2
ExpReturn = 0.05

# Simulation parameters
rfRate = 0
dT = 1/250
nSteps = Maturity/dT
nTrials = 5000

# Transacation cost and cost function parameters
c = 1.5
kappa = .01
InitPosition = 0

# Set the random generator seed for reproducibility.
random.seed(3)


Local functions 

Define Enviroment
  
UpperLimit, [10 Maturity 1]': This sets specific maximums for each of the three variables:
-Moneyness: Max 10 (Spot price is 10x the Strike).
-TimeToMaturity: Max is the Maturity variable (time doesn't go backward).
-Position: Max 1 (You can't hold more than 100% of the underlying asset in this specific model).

Action space: LowerLimit, 0 and UpperLimit, 1: The action must be between 0 and 1. In this hedging example, the action represents the Hedge Ratio (how much of the stock to hold). 0 means hold nothing, 1 means hold 100% of the shares required to cover the option.

In [6]:
# We define the lower and upper bounds as arrays
low = np.array([0, 0, 0], dtype=np.float32)
high = np.array([10, Maturity, 1], dtype=np.float32)

# 'Box' is the Gymnasium equivalent of 'rlNumericSpec' for continuous values
observation_space = spaces.Box(low=low, high=high, shape=(3,), dtype=np.float32)
# Custom labeling 
observation_labels = ['Moneyness', 'TimeToMaturity', 'Position']

# A single continuous value between 0 and 1
action_space = spaces.Box(low=0, high=1, shape=(1,), dtype=np.float32)

print(action_space)


Box(0.0, 1.0, (1,), float32)


In [ ]:

class HedgingEnv(gym.Env):
    
    def __init__(self, maturity=1.0, dt=1/252):
        super(HedgingEnv, self).__init__()
        self.dt = dt
        self.maturity = maturity

        # We define the lower and upper bounds as arrays
        low = np.array([0, 0, 0], dtype=np.float32)
        high = np.array([10, Maturity, 1], dtype=np.float32)

        # Custom labeling 
        self.observation_labels = ['Moneyness', 'TimeToMaturity', 'Position']

        # Define spaces (as we discussed previously)
        self.observation_space = spaces.Box(low=low, high=high, shape=(3,), dtype=np.float32)
        self.action_space = spaces.Box(low=0, high=1, shape=(1,), dtype=np.float32)

    def step(self, action):
        # 1. Unpack current state (MATLAB: LoggedSignals.State)
        # state = [Moneyness, TimeToMaturity, PrevPosition]
        moneyness, ttm, prev_pos = self.state
        
        # 2. Get the new position from the agent (Action)
        # Note: action is a numpy array from Gymnasium
        new_pos = action[0] 
        
        # 3. Simulate Price Movement (Geometric Brownian Motion)
        # This matches the transition logic in the MATLAB loop
        vol = 0.2  # Annualized volatility
        z = np.random.normal()
        # dS/S = vol * sqrt(dt) * Z
        price_ratio = np.exp(-0.5 * vol**2 * self.dt + vol * np.sqrt(self.dt) * z)
        new_moneyness = moneyness * price_ratio
        
        # 4. Calculate Costs (The MATLAB reward logic)
        # Transaction costs: κ * |ΔPosition| * Price
        cost_multiplier = 0.01 # Example: 1% transaction cost
        transaction_costs = cost_multiplier * np.abs(new_pos - prev_pos) * new_moneyness
        
        # Hedging Error (Tracking Error)
        # This is the change in value of the portfolio minus the change in option value
        # Simplified here as the variation in the hedged position:
        wealth_change = prev_pos * (new_moneyness - moneyness)
        
        # Reward is negative cost (we want to maximize, so we minimize expenses)
        reward = -(transaction_costs + np.abs(wealth_change))
        
        # 5. Update Time and check for termination
        new_ttm = ttm - self.dt
        terminated = bool(new_ttm <= 0)
        truncated = False # Typically used for time limits unrelated to the physics
        
        # 6. Update internal state
        self.state = np.array([new_moneyness, new_ttm, new_pos], dtype=np.float32)
        
        return self.state, float(reward), terminated, truncated, {}

In [ ]:
import numpy as np
from scipy.stats import norm

class HedgingEnv:
    def __init__(self, strike, r, vol, maturity, dt, kappa, c, moneyness, time_to_maturity, init_position):
        self.K = strike 
        self.r = r
        self.vol = vol
        self.T = maturity
        self.dt = dt
        self.kappa = kappa # Transaction cost rate
        self.c = c         # Risk aversion penalty
        self.state = moneyness, time_to_maturity, init_position  # Create the state as a 1D array (equivalent to the column vector)
        
        

    def reset_fcn(self):
        
        # In Python, we usually return the observation and an empty info dict
        # to match the Gymnasium API standard.
        logged_signals = {"State": self.state}
        initial_observation = logged_signals
        
        return initial_observation, logged_signals

            
    def get_option_value(self, S, t):
        """Standard Black-Scholes Call Price (V)"""
        if t <= 0: return max(S - self.K, 0)
        d1 = (np.log(S/self.K) + (self.r + 0.5*self.vol**2)*t) / (self.vol*np.sqrt(t))
        d2 = d1 - self.vol*np.sqrt(t)
        return S * norm.cdf(d1) - self.K * np.exp(-self.r * t) * norm.cdf(d2)

    def step(self, action):
        # Current state before step
        moneyness, ttm, h_prev = self.state
        s_curr = moneyness * self.K # S_i
        v_curr = self.get_option_value(s_curr, ttm) # V_i
        
        # 1. Update the Market (S_{i+1})
        # Note: In RL finance, we often use ExpReturn (mu) for price simulation
        mu = 0.05 
        z = np.random.standard_normal()
        s_next = s_curr * np.exp((mu - 0.5 * self.vol**2) * self.dt + self.vol * np.sqrt(self.dt) * z)
        ttm_next = max(0, ttm - self.dt)
        v_next = self.get_option_value(s_next, ttm_next) # V_{i+1}
        
        # 2. Agent Action (H_{i+1})
        h_next = action[0] 
        
        # 3. Calculate Raw Reward (R_{i+1} P&L Formulation)
        # R = (V_next - V_curr) + H_next*(S_next - S_curr) - kappa * |S_next * (H_next - H_prev)|
        delta_v = v_next - v_curr
        delta_s = s_next - s_curr
        trans_cost = self.kappa * np.abs(s_next * (h_next - h_prev))
        
        raw_reward = delta_v + h_next * delta_s - trans_cost
        
        # 4. Final Step Liquidation Cost
        # At maturity, we must close the hedge: kappa * |S_n * H_n|
        if ttm_next <= 0:
            liquidation_cost = self.kappa * np.abs(s_next * h_next)
            raw_reward -= liquidation_cost
            
        # 5. Apply Risk Penalty: R = R - c(R^2)
        # This penalizes high volatility in P&L
        penalized_reward = raw_reward - self.c * (raw_reward**2)
        
        # Update state
        self.state = np.array([s_next/self.K, ttm_next, h_next], dtype=np.float32)
        
        terminated = bool(ttm_next <= 0)
        return self.state, penalized_reward, terminated, False, {}